# NYC Elevator Complaints — City-Wide Overview

> **Narrative:** *"This isn't a District 17 problem. It's a city-wide failure of accountability — and the data proves it."*

Source: [NYC Open Data — DOB Elevator Complaint Dispositions](https://data.cityofnewyork.us/Housing-Development/DOB-Elevator-Complaint-Dispositions/kqwi-7ncn)  
Codes: `6S` (elevator complaint) · `6M` (elevator/escalator complaint)

In [ ]:
# ── PARAMETERS — edit before presenting ──────────────────────────
YEARS = [2025, 2026]   # years to include
TOP_N = 20             # buildings on the leaderboard
# ─────────────────────────────────────────────────────────────────

import os, requests
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../../.env"))
except ImportError:
    pass

TOKEN = os.getenv("SODA_APP_TOKEN")
if not TOKEN:
    print("⚠  SODA_APP_TOKEN not set — add it to .env or export it in your shell")

SODA_URL  = "https://data.cityofnewyork.us/resource/kqwi-7ncn.json"
BOROUGH   = {"1": "Manhattan", "2": "Bronx", "3": "Brooklyn", "4": "Queens", "5": "Staten Island"}
YEAR_FILTER = "(" + " OR ".join(f"date_entered LIKE '%/{y}'" for y in YEARS) + ")"
YEAR_LABEL  = " + ".join(str(y) for y in sorted(YEARS))

def _get(params: dict) -> list:
    if TOKEN:
        params["$$app_token"] = TOKEN
    r = requests.get(SODA_URL, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f5f5f5",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "sans-serif",
    "axes.titlesize":   14,
    "axes.labelsize":   11,
})

print(f"Parameters: years={YEAR_LABEL}  top_n={TOP_N}")

In [ ]:
print(f"Fetching city-wide data for {YEAR_LABEL}...")

# Total complaint count
total_data = _get({"$select": "count(*) AS total",
                   "$where": f"complaint_category IN ('6S','6M') AND {YEAR_FILTER}"})
total = int(total_data[0]["total"])

# Borough breakdown
borough_rows = _get({"$select": "community_board, count(*) AS cnt",
                     "$where": f"complaint_category IN ('6S','6M') AND {YEAR_FILTER}",
                     "$group": "community_board",
                     "$limit": "500"})
borough_totals: dict[str, int] = defaultdict(int)
for row in borough_rows:
    cb = row.get("community_board", "")
    b  = BOROUGH.get(cb[0], "Unknown") if cb else "Unknown"
    borough_totals[b] += int(row.get("cnt", 0))

# Top buildings
top_rows = _get({"$select": "house_number, house_street, community_board, count(*) AS complaint_count",
                 "$where": f"complaint_category IN ('6S','6M') AND {YEAR_FILTER}",
                 "$group": "house_number, house_street, community_board",
                 "$order": "complaint_count DESC",
                 "$limit": str(TOP_N)})

print(f"✓  {total:,} total complaints | {len(top_rows)} top buildings loaded")

## Headline Stat

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.axis("off")
ax.text(0.5, 0.65, f"{total:,}",
        ha="center", va="center", fontsize=56, fontweight="bold", color="#1a1a2e",
        transform=ax.transAxes)
ax.text(0.5, 0.18, f"elevator complaints filed in NYC ({YEAR_LABEL})",
        ha="center", va="center", fontsize=13, color="#555",
        transform=ax.transAxes)
fig.tight_layout()
plt.show()

## Complaints by Borough

In [ ]:
borough_order = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
counts = [borough_totals.get(b, 0) for b in borough_order]
pcts   = [c / total * 100 if total else 0 for c in counts]

colors = ["#e63946" if b in ("Bronx", "Brooklyn") else "#457b9d" for b in borough_order]

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(borough_order, counts, color=colors, height=0.55, edgecolor="white")

for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{count:,}  ({pct:.1f}%)", va="center", fontsize=10, color="#333")

ax.set_xlabel("Complaint count")
ax.set_title(f"Elevator Complaints by Borough — {YEAR_LABEL}", pad=12)
ax.set_xlim(0, max(counts) * 1.22)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
plt.show()

## Top Buildings City-Wide

In [ ]:
rows = []
for i, r in enumerate(top_rows, 1):
    cb      = r.get("community_board", "")
    borough = BOROUGH.get(cb[0], "?")[:10] if cb else "?"
    addr    = r.get("house_number", "") + " " + r.get("house_street", "")
    count   = int(r.get("complaint_count", 0))
    rows.append({"Rank": i, "Address": addr, "Borough": borough, "CB": cb, "Complaints": count})

df = pd.DataFrame(rows)

# Bar chart
fig, ax = plt.subplots(figsize=(9, TOP_N * 0.38 + 1))
bar_colors = ["#e63946" if r["Borough"] in ("Bronx", "Brooklyn") else "#457b9d" for _, r in df.iterrows()]
ax.barh(df["Address"] + "  (" + df["Borough"] + ")",
        df["Complaints"], color=bar_colors, height=0.6, edgecolor="white")

for i, (count, bar) in enumerate(zip(df["Complaints"], ax.patches)):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=9, color="#333")

ax.set_xlabel("Complaint count")
ax.set_title(f"Top {TOP_N} Buildings — NYC ({YEAR_LABEL})", pad=12)
ax.invert_yaxis()
fig.tight_layout()
plt.show()

# Print table
print(df.to_string(index=False))